# Variables antitéticas y valuación de una Call europea
##### María Fernanda Gómez Flores
##### Lunes 19 de octubre de 2026

**Simulación de Procesos Financieros**

## Parte 1. Distribución normal estándar

Primero voy a sacar numeros de una normal $N(0,1)$ y separarlos entre positivos y negativos para ver cuanto da el promedio de cada lado

### Importar librerías

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

### Simulación de valores normales

In [2]:
M_normal = 100  # Cantidad usada en mi notebook original
Z_normal = np.random.normal(0, 1, M_normal)

arriba = Z_normal[Z_normal > 0]
abajo = Z_normal[Z_normal < 0]

print("Cantidad arriba:", len(arriba))
print("Cantidad abajo:", len(abajo))
print("Promedio arriba:", np.mean(arriba))
print("Promedio abajo:", np.mean(abajo))

Cantidad arriba: 53
Cantidad abajo: 47
Promedio arriba: 0.788552644034983
Promedio abajo: -0.7607695591718642


Como los numeros son aleatorios los promedios no siempre van a salir iguales, pero deberian quedar mas o menos cerca de 0.7979 para los positivos y -0.7979 para los negativos

## Parte 2. Precio de una Call europea con Black-Scholes

Ahora voy a calcular la call con Black-Scholes para tener un valor con el que pueda comparar las simulaciones

$$d_1=\frac{\ln(S_0/K)+(r+\sigma^2/2)T}{\sigma\sqrt{T}},\quad d_2=d_1-\sigma\sqrt{T}$$

$$C_0=S_0N(d_1)-Ke^{-rT}N(d_2)$$

### Parámetros

In [3]:
S0 = 100
r = 0.05
sigma = 0.20
T = 1
K = 100
n = 252
M = 100000

### Cálculo de Black-Scholes

In [4]:
d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
d2 = d1 - sigma*np.sqrt(T)

VBS = S0*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

print("d1:", round(d1, 4))
print("d2:", round(d2, 4))
print("Valor Call Black-Scholes:", round(VBS, 4))

d1: 0.35
d2: 0.15
Valor Call Black-Scholes: 10.4506


Este es el precio teorico de la call y aqui no uso ni n ni M porque todavia no estoy haciendo simulaciones

## Parte 3. Payoff con $Z=1$ y $Z=-1$

Aqui voy a usar $dt=1$ y calcular que pasa con la call cuando Z vale 1 y cuando vale -1

$$S_T=S_0e^{(r-\sigma^2/2)dt+\sigma\sqrt{dt}Z}$$

$$V_T=\max(S_T-K,0),\qquad VA=V_Te^{-r\,dt}$$

### Escenario 1. $Z=1$

In [5]:
dt = 1
Z = 1
ST = S0*np.exp((r-0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z)
payoff = max(ST-K, 0)
VA = payoff*np.exp(-r*dt)

print("Precio final:", round(ST, 4))
print("Payoff Call:", round(payoff, 4))
print("Valor actual Call:", round(VA, 4))

Precio final: 125.86
Payoff Call: 25.86
Valor actual Call: 24.5988


### Escenario 2. $Z=-1$

In [6]:
Z = -1
ST = S0*np.exp((r-0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z)
payoff = max(ST-K, 0)
VA = payoff*np.exp(-r*dt)

print("Precio final:", round(ST, 4))
print("Payoff Call:", round(payoff, 4))
print("Valor actual Call:", round(VA, 4))

Precio final: 84.3665
Payoff Call: 0
Valor actual Call: 0.0


### Promedio de ambos payoffs

Ahora saco el promedio de los dos payoffs con la formula en vez de escribirlo a mano

In [7]:
ST_positivo = S0*np.exp((r-0.5*sigma**2)*dt + sigma*np.sqrt(dt)*1)
ST_negativo = S0*np.exp((r-0.5*sigma**2)*dt - sigma*np.sqrt(dt)*1)

promedio_dos = (max(ST_positivo-K, 0) + max(ST_negativo-K, 0))/2
print("Promedio de los dos payoffs:", round(promedio_dos, 4))

Promedio de los dos payoffs: 12.93


Con Z=1 y Z=-1 estoy usando dos escenarios contrarios, saco el payoff de cada uno y despues el promedio, este todavia es el valor al final del periodo

## Parte 4. Estimación con 10 valores de Z y variables antitéticas

Ahora uso los 10 valores de Z del ejercicio y para cada uno voy a calcular el precio final y el payoff, despues hago lo mismo pero con -Z y saco el promedio de los dos

$$\overline{V}_i=\frac{V(Z_i)+V(-Z_i)}{2}$$

### Cálculo de precios y payoffs

In [8]:
Z = np.array([0.07629, -0.2985, -0.7983, 0.993, -1.3465,
              1.2462, -1.1405, 0.7615, 0.2816, 0.81])

ST_z = S0*np.exp((r-0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z)
ST_menosZ = S0*np.exp((r-0.5*sigma**2)*dt - sigma*np.sqrt(dt)*Z)

V_Z = np.maximum(ST_z-K, 0)
V_menosZ = np.maximum(ST_menosZ-K, 0)
V_promedio = (V_Z + V_menosZ)/2

### Tabla de resultados

In [9]:
tabla = pd.DataFrame({
    "Z": Z,
    "V_z": V_Z,
    "V_-z": V_menosZ,
    "Promedio": V_promedio
})

tabla = tabla.round(4)
tabla

,Z,V_z,V_-z,Promedio
0,0.0763,4.6298,1.4851,3.0574
1,-0.2985,0.0000,9.3846,4.6923
2,-0.7983,0.0000,20.8839,10.4419
3,0.9930,25.6839,0.0000,12.8420
4,-1.3465,0.0000,34.8914,17.4457
5,1.2462,32.2125,0.0000,16.1062
6,-1.1405,0.0000,29.4468,14.7234
7,0.7615,19.9974,0.0000,9.9987
8,0.2816,9.0155,0.0000,4.5078
9,0.8100,21.1671,0.0000,10.5835


### Estimaciones de los promedios

In [13]:
estimacion_Z = np.mean(V_Z)
estimacion_menosZ = np.mean(V_menosZ)
estimacion_antitetica = np.mean(V_promedio)

print("Estimación usando solamente Z:", round(estimacion_Z, 4))
print("Estimación usando solamente -Z:", round(estimacion_menosZ, 4))
print("Estimación antitética:", round(estimacion_antitetica, 4))

Estimación usando solamente Z: 11.2706
Estimación usando solamente -Z: 9.6092
Estimación antitética: 10.4399


**Interpretación:** usando solo Z me da una estimacion y usando -Z me da otra, al sacar el promedio de las dos estoy usando variables antiteticas para intentar reducir la varianza de la simulacion

## Conclusión

Con Black-Scholes ya tengo el precio teorico de la call y con la simulacion puedo aproximarlo. Como aqui solo use 10 valores puede haber diferencia entre los resultados, con las variables antiteticas uso Z y -Z para tratar de reducir la variacion aunque no siempre quiere decir que me vaya a quedar mas cerca de Black-Scholes